In [11]:
import pandas as pd
import numpy as np
import re

In [12]:
# 1. Завантажуємо оригінальний тренувальний файл SNLI
df_full = pd.read_csv('../data/snli_1.0_train.txt', sep='\t', on_bad_lines='skip')

# 2. Залишаємо тільки потрібні колонки (лейбл, передумова, гіпотеза)
df_full = df_full[['gold_label', 'sentence1', 'sentence2']]
df_full.columns = ['label', 'premise', 'hypothesis']

# 3. Відкидаємо рядки без консенсусу (лейбл '-') та пусті значення
df_full = df_full[df_full['label'].isin(['entailment', 'contradiction', 'neutral'])]
df_full = df_full.dropna()

# 4. Робимо рівномірну вибірку: по 500 прикладів на клас (разом 1500)
df_sampled = df_full.groupby('label').sample(n=500, random_state=42).reset_index(drop=True)

# 5. Зберігаємо як наш raw.csv
df_sampled.to_csv('../data/raw.csv', index=False)
print(f"Збережено raw.csv. Розмір: {df_sampled.shape}")

Збережено raw.csv. Розмір: (1500, 3)


In [13]:
# Завантажуємо наш підготовлений датасет
df = pd.read_csv('../data/raw/raw.csv')

In [14]:
print("1. Десять прикладів даних")
display(df.head(10))

1. Десять прикладів даних


,label,premise,hypothesis
0,contradiction,A woman and a young girl smiling for the camer...,the mom and daughter are very sad at the dog d...
1,contradiction,"A young boy wearing a gray sweater, blue jeans...",The boy is sitting down.
2,contradiction,A woman with a very large black wig and giant ...,The woman is cooking breakfast inside
3,contradiction,Oriental man dressed in tank top and shorts is...,the man is white
4,contradiction,3 girls and one boy playing in the street.,The children are all indoors.
5,contradiction,Two girls are going for a swim in a mountain l...,A boy and a girl are having a romantic swim in...
6,contradiction,A woman fiddles with her phone at a diner.,she is swimming the english channel
7,contradiction,Two women are walking casually down the street...,There are two men
8,contradiction,A bunch of kids in canoes on a river.,A few adults are swimming in the lake.
9,contradiction,A white dog and two black dogs playing,Two cats playing.


In [15]:
print("2. Статистика датасету")
print(f"Загальна кількість текстів (пар): {len(df)}")
print("\nРозподіл класів:")
print(df['label'].value_counts())

# Рахуємо довжину символів та слів для Premise і Hypothesis
for col in ['premise', 'hypothesis']:
    char_len = df[col].astype(str).apply(len)
    word_len = df[col].astype(str).apply(lambda x: len(x.split()))
    print(f"\n{col.capitalize()} - Медіанна довжина (символи): {char_len.median():.1f}")
    print(f"{col.capitalize()} - Медіанна довжина (слова): {word_len.median():.1f}")

2. Статистика датасету
Загальна кількість текстів (пар): 1500

Розподіл класів:
label
contradiction    500
entailment       500
neutral          500
Name: count, dtype: int64

Premise - Медіанна довжина (символи): 60.0
Premise - Медіанна довжина (слова): 12.0

Hypothesis - Медіанна довжина (символи): 34.0
Hypothesis - Медіанна довжина (слова): 7.0


In [16]:
# 3. Базова нормалізація
def normalize_text(text):
    text = str(text)
    # Заміна URL
    text = re.sub(r'http\S+|www\.\S+', '<URL>', text)
    # Заміна Email
    text = re.sub(r'\S+@\S+', '<EMAIL>', text)
    # Заміна телефонів
    text = re.sub(r'\+?\d[\d -]{8,12}\d', '<PHONE>', text)
    # Уніфікація апострофів
    text = re.sub(r"[‘’`´]", "'", text)
    # Прибирання зайвих пробілів та переносів
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [17]:
df['premise_clean'] = df['premise'].apply(normalize_text)
df['hypothesis_clean'] = df['hypothesis'].apply(normalize_text)

print("Нормалізацію завершено. Приклад:")
display(df[['premise', 'premise_clean']].head(5))

Нормалізацію завершено. Приклад:


,premise,premise_clean
0,A woman and a young girl smiling for the camer...,A woman and a young girl smiling for the camer...
1,"A young boy wearing a gray sweater, blue jeans...","A young boy wearing a gray sweater, blue jeans..."
2,A woman with a very large black wig and giant ...,A woman with a very large black wig and giant ...
3,Oriental man dressed in tank top and shorts is...,Oriental man dressed in tank top and shorts is...
4,3 girls and one boy playing in the street.,3 girls and one boy playing in the street.


In [18]:
# 4. Перевірки якості

# Точні дублікати за парами
duplicates_pct = df.duplicated(subset=['premise_clean', 'hypothesis_clean']).mean() * 100
print(f"Відсоток точних дублікатів: {duplicates_pct:.2f}%")

# Дуже короткі рядки
short_premise = df['premise_clean'].apply(lambda x: len(x.split()) < 5).sum()
short_hypothesis = df['hypothesis_clean'].apply(lambda x: len(x.split()) < 5).sum()
print(f"Короткі передумови (< 5 слів): {short_premise}")
print(f"Короткі гіпотези (< 5 слів): {short_hypothesis}")

Відсоток точних дублікатів: 0.00%
Короткі передумови (< 5 слів): 15
Короткі гіпотези (< 5 слів): 226


In [19]:
# Сміттєві рядки
def is_garbage(text):
    return not bool(re.search(r'[a-zA-Z]', text))

garbage_premise = df['premise_clean'].apply(is_garbage).sum()
garbage_hypothesis = df['hypothesis_clean'].apply(is_garbage).sum()
print(f"Сміттєві передумови: {garbage_premise}")
print(f"Сміттєві гіпотези: {garbage_hypothesis}")

Сміттєві передумови: 0
Сміттєві гіпотези: 0


In [20]:
# Зберігаємо оброблений датасет
df.to_csv('../data/processed/processed.csv', index=False)
print("\nДатасет збережено у /data/processed/processed.csv")


Датасет збережено у /data/processed/processed.csv


### 5. Висновок
Датасет SNLI є високоякісним джерелом для задачі Natural Language Inference (NLI). Після відбору 1500 пар речень ми маємо ідеально збалансований розподіл (по 500 прикладів на класи `entailment`, `contradiction`, `neutral`). Аналіз показав, що медіанна довжина гіпотез очікувано менша за передумови, оскільки вони є короткими логічними наслідками. У даних відсутні "сміттєві" рядки (без літер) та критично короткі передумови. Головним ризиком є наявність дуже коротких гіпотез (< 5 слів), які можуть нести недостатньо контексту для класичних моделей ML. У наступних кроках (Lab 2) доцільно провести більш глибоку лематизацію, видалення стоп-слів та векторизацію для підготовки тексту до навчання.